In [ ]:
# =========================================
# ROAD ACCIDENT SEVERITY PREDICTION SYSTEM
# =========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

# ==========================
# LOAD DATASET
# ==========================

df = pd.read_csv("road_accident_data.csv")
print(df.head())

# ==========================
# DATA CLEANING
# ==========================

df.drop_duplicates(inplace=True)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# ==========================
# DROP LEAKAGE / NON-PREDICTIVE COLUMNS
# ==========================
# - accident_id: row identifier, no real-world meaning
# - casualties: directly defines/encodes severity (e.g. fatal = casualties > 0) -> leakage
# - risk_score: pre-computed score likely derived from severity -> leakage
# - date: label-encoding a date string is meaningless; day_of_week/hour/is_weekend
#         already capture the useful temporal signal
LEAKAGE_COLS = ["accident_id", "casualties", "risk_score", "date"]
df = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns])

# NOTE: Verify against your data dictionary that 'casualties' and 'risk_score'
# are not needed/safe. If risk_score is computed ONLY from pre-crash conditions
# (weather, traffic, road type etc.) and not from the outcome, it could be kept -
# but as supplied with no documentation, treat it as leakage.

# ==========================
# FEATURE / TARGET SPLIT
# ==========================

target = "accident_severity"

X = df.drop(columns=[target])
y = df[target]

# ==========================
# ENCODE CATEGORICAL COLUMNS
# ==========================

categorical_cols = X.select_dtypes(include=['object']).columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

# ==========================
# TRAIN/TEST SPLIT (BEFORE SMOTE)
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================
# FEATURE SCALING (fit on train only)
# ==========================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================
# HANDLE IMBALANCED DATA (SMOTE ON TRAIN ONLY)
# ==========================

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("Train class distribution after SMOTE:")
print(pd.Series(y_train_res).value_counts())
print("\nTest class distribution (untouched, real-world):")
print(pd.Series(y_test).value_counts())

# ==========================
# BASELINE (MAJORITY CLASS)
# ==========================

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train_res, y_train_res)
dummy_pred = dummy.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, dummy_pred)

# ==========================
# MODEL 1: RANDOM FOREST
# ==========================

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42
)
rf_model.fit(X_train_res, y_train_res)
rf_pred = rf_model.predict(X_test_scaled)

# ==========================
# MODEL 2: XGBOOST
# ==========================

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train_res, y_train_res)
xgb_pred = xgb_model.predict(X_test_scaled)

# ==========================
# MODEL 3: ADABOOST + DECISION TREE
# ==========================

dt = DecisionTreeClassifier(max_depth=5)

ada_model = AdaBoostClassifier(
    estimator=dt,
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)
ada_model.fit(X_train_res, y_train_res)
ada_pred = ada_model.predict(X_test_scaled)

# ==========================
# EVALUATION FUNCTION
# ==========================

def evaluate_model(name, y_true, y_pred):
    print("\n==============================")
    print(f"MODEL: {name}")
    print("==============================")

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return accuracy

# ==========================
# RESULTS
# ==========================

print("\n==============================")
print("BASELINE (majority class)")
print("==============================")
print(f"Accuracy  : {baseline_acc:.4f}")

rf_acc = evaluate_model("Random Forest", y_test, rf_pred)
xgb_acc = evaluate_model("XGBoost", y_test, xgb_pred)
ada_acc = evaluate_model("AdaBoost", y_test, ada_pred)

accuracies = {
    "Baseline": baseline_acc,
    "Random Forest": rf_acc,
    "XGBoost": xgb_acc,
    "AdaBoost": ada_acc
}

best_model = max(
    (k for k in accuracies if k != "Baseline"),
    key=accuracies.get
)

print("\n================================")
print("BEST MODEL:", best_model)
print("BEST ACCURACY:", accuracies[best_model])
print("================================")

# ==========================
# VISUALIZATION: MODEL COMPARISON
# ==========================

models = list(accuracies.keys())
scores = list(accuracies.values())

plt.figure(figsize=(8, 5))
plt.bar(models, scores)
plt.title("Model Accuracy Comparison (Baseline vs Models)")
plt.ylabel("Accuracy")
plt.xlabel("Models")

for i, v in enumerate(scores):
    plt.text(i, v + 0.01, str(round(v, 3)), ha='center')

plt.show()

# ==========================
# FEATURE IMPORTANCE (Random Forest)
# ==========================

feature_importance = rf_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

print("\nTop Important Features:")
print(importance_df.head(10))

plt.figure(figsize=(10, 6))
plt.barh(
    importance_df['Feature'][:10],
    importance_df['Importance'][:10]
)
plt.xlabel("Importance")
plt.title("Top 10 Important Features")
plt.gca().invert_yaxis()
plt.show()
